# Exploration — Audiogrammes Odyo / Corti

Ce notebook explore les données brutes et teste le pipeline non supervisé.

**Étapes :**
1. Chargement des données JSON
2. Exploration descriptive (distribution, catégories de visite)
3. Features absolues (seuils interpolés, PTA, asymétrie)
4. Features delta (évolution par patient vs Baseline)
5. Pipeline non supervisé (Isolation Forest + Autoencoder + PCA)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loader import load_json_file, load_dataset
from src.features import (
    build_feature_matrix,
    build_delta_features,
    preprocess,
    STANDARD_FREQS,
    VISIT_CATEGORY_LABELS,
)
from src.models.unsupervised import run_unsupervised_pipeline
from src.evaluate import (
    plot_anomaly_score_distribution,
    plot_audiogram,
    plot_top_anomalies,
    plot_umap,
    plot_patient_trajectory,
    plot_delta_heatmap,
    plot_sts_distribution,
    summary_report,
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Chargement des données

In [ ]:
# Option A : charger un seul fichier (sample)
records = load_json_file('../sample.json')
df = pd.DataFrame(records)

# Option B : charger tout un dossier
# df = load_dataset('../data/raw/')

print(f"Records chargés : {len(df)}")
df.head()

## 2. Exploration descriptive

In [ ]:
print("=== Colonnes ===")
print(df.dtypes)

print("\n=== Valeurs manquantes ===")
print(df.isnull().sum())

print("\n=== Distribution des catégories de visite ===")
visit_counts = df['visit_category'].value_counts().sort_index()
for cat, count in visit_counts.items():
    label = VISIT_CATEGORY_LABELS.get(cat, '?')
    print(f"  {cat} ({label:10s}) : {count} records")

print("\n=== Patients uniques ===")
print(f"  {df['patient'].nunique()} patients")

print("\n=== Patients avec Baseline ===")
has_baseline = df[df['visit_category'] == 0]['patient'].unique()
print(f"  {len(has_baseline)} / {df['patient'].nunique()} patients ont un Baseline")

In [ ]:
# Visualiser un audiogramme exemple
row = df.iloc[0]
visit_label = VISIT_CATEGORY_LABELS.get(row['visit_category'], '?')
plot_audiogram(
    row['dots_left'],
    row['dots_right'],
    title=f"Exemple — {visit_label} | record #{row['record_id'][:12]}..."
)
plt.show()

In [ ]:
# Distribution du nombre de fréquences testées
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['n_freqs_left'].hist(ax=axes[0], bins=15, color='blue', alpha=0.7)
axes[0].set_title('Nb fréquences OG')
df['n_freqs_right'].hist(ax=axes[1], bins=15, color='red', alpha=0.7)
axes[1].set_title('Nb fréquences OD')
plt.tight_layout()
plt.show()

In [ ]:
# Trajectoire d'un patient (si plusieurs visites disponibles)
# Choisir un patient avec plusieurs records
patients_multi = df.groupby('patient').filter(lambda x: len(x) > 1)['patient'].unique()

if len(patients_multi) > 0:
    plot_patient_trajectory(df, patients_multi[0])
    plt.show()
else:
    print("Pas assez de records par patient pour afficher une trajectoire.")

## 3. Features absolues

In [ ]:
feature_df, feature_cols = build_feature_matrix(df)
X_abs, scaler_abs, imputer_abs = preprocess(feature_df, fit=True)

print(f"Shape matrice absolue : {X_abs.shape}")
print(f"Features : {feature_cols}")
feature_df.describe().round(1)

In [ ]:
# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    feature_df.corr(), annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Corrélation entre features absolues')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des seuils par fréquence
left_cols = [f'L_{f}' for f in STANDARD_FREQS]
right_cols = [f'R_{f}' for f in STANDARD_FREQS]

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, (lc, rc, freq) in enumerate(zip(left_cols, right_cols, STANDARD_FREQS)):
    feature_df[lc].hist(ax=axes[0, i], bins=20, color='blue', alpha=0.7)
    axes[0, i].set_title(f'OG {freq} Hz')
    feature_df[rc].hist(ax=axes[1, i], bins=20, color='red', alpha=0.7)
    axes[1, i].set_title(f'OD {freq} Hz')
plt.suptitle('Distribution des seuils dB par fréquence', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Features delta (évolution par patient vs Baseline)

Pour chaque record **Periodic** ou **Depart**, on calcule le changement fréquence par fréquence
par rapport au **Baseline** du même patient.

→ Plus robuste cliniquement : on cherche des *changements anormaux*, pas des seuils absolus élevés.

In [ ]:
delta_df = build_delta_features(df)

n_with_delta = delta_df['delta_PTA_L'].notna().sum()
print(f"Records avec delta calculable : {n_with_delta} / {len(df)}")
print(f"(Baseline et patients sans Baseline = NaN)")

delta_df.describe().round(2)

In [ ]:
# Distribution des deltas PTA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['delta_PTA_L', 'delta_PTA_R'], ['OG (gauche)', 'OD (droite)']):
    values = delta_df[col].dropna()
    if len(values) > 0:
        ax.hist(values, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
        ax.axvline(0, color='gray', linestyle=':', alpha=0.7)
        ax.axvline(10, color='red', linestyle='--', alpha=0.8, label='STS ≥ 10 dB')
        ax.set_title(f'Delta PTA {label}')
        ax.set_xlabel('Δ dB vs Baseline (+ = aggravation)')
        ax.legend()
    else:
        ax.set_title(f'Delta PTA {label} — pas de données')
plt.suptitle('Évolution du Pure Tone Average vs Baseline', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Standard Threshold Shift
plot_sts_distribution(delta_df)
plt.show()

# Résumé STS
for col, label in [('has_sts_L', 'OG'), ('has_sts_R', 'OD')]:
    if col in delta_df.columns:
        n_sts = delta_df[col].sum()
        n_total = delta_df[col].notna().sum()
        if n_total > 0:
            print(f"STS {label} (≥10 dB) : {n_sts:.0f} / {n_total} ({100*n_sts/n_total:.1f}%)")

In [ ]:
# Heatmap des deltas fréquentiels
if delta_df['delta_L_250'].notna().any():
    plot_delta_heatmap(delta_df, df=df, ear='L')
    plt.show()
    plot_delta_heatmap(delta_df, df=df, ear='R')
    plt.show()
else:
    print("Pas de records delta disponibles (besoin de patients avec Baseline + Periodic/Depart).")

In [ ]:
# Préparer la matrice delta pour le pipeline (uniquement records avec delta)
delta_mask = delta_df['delta_PTA_L'].notna()
print(f"Records utilisables pour pipeline delta : {delta_mask.sum()}")

if delta_mask.sum() >= 5:
    X_delta, scaler_delta, imputer_delta = preprocess(delta_df[delta_mask], fit=True)
    print(f"Shape matrice delta : {X_delta.shape}")
else:
    print("Pas assez de records avec delta — utiliser les features absolues pour le pipeline.")
    X_delta = None

## 5. Pipeline non supervisé

Deux modes :
- **Absolu** (`X_abs`) : détecte les audiogrammes avec une configuration inhabituelle en valeur absolue
- **Delta** (`X_delta`) : détecte les patients dont l'évolution vs Baseline est anormale *(préféré cliniquement)*

In [ ]:
# Choisir la matrice ici :
# X = X_abs   # features absolues (tous les records)
# X = X_delta # features delta (uniquement Periodic/Depart avec Baseline)

X = X_abs
df_pipeline = df  # aligner avec X

scores_df, if_model, ae_model = run_unsupervised_pipeline(
    X,
    contamination=0.05,
    ae_epochs=100,
)

summary_report(df_pipeline, scores_df)

In [ ]:
# Distribution des scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_anomaly_score_distribution(scores_df['anomaly_score_if'], title='Isolation Forest', ax=axes[0])
plot_anomaly_score_distribution(scores_df['reconstruction_error'], title='Autoencoder', ax=axes[1])
plot_anomaly_score_distribution(scores_df['pca_reconstruction_error'], title='PCA baseline', ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloré par score d'anomalie
plot_umap(
    X,
    scores_df['reconstruction_error'].values,
    title='UMAP — Erreur de reconstruction (Autoencoder)',
    label_name='Reconstruction error'
)
plt.show()

# UMAP coloré par catégorie de visite
plot_umap(
    X,
    df_pipeline['visit_category'].values,
    title='UMAP — Catégorie de visite',
    label_name='0=Baseline 1=Periodic 2=Depart',
    is_categorical=True,
)
plt.show()

In [ ]:
# Top anomalies (consensus)
plot_top_anomalies(
    df_pipeline,
    scores_df['reconstruction_error'],
    n=6,
    score_col='reconstruction_error'
)
plt.show()

In [ ]:
# Trajectoires des patients les plus anormaux (consensus)
anomaly_patients = (
    df_pipeline[scores_df['anomaly_consensus'] == 1]['patient']
    .unique()
)

for patient in anomaly_patients[:3]:
    plot_patient_trajectory(
        df,
        patient,
        score_col='reconstruction_error',
        scores_df=scores_df,
    )
    plt.show()